# Лабораторная работа №3: Промпты и температура

[Исходное задание](https://colab.research.google.com/drive/1XAScNXarTrsV1vDoDG24EdkQtpXT80uI)

Выполнение через совместимый API RouterAI. API-ключ берётся из окружения или локального `.env`. Сохранённые результаты получены реальными запросами.



## Задание

#Задание семинара №3
1. Задайте вопрос LLM по взятой теме при 3-х разных температурах (низкая, средняя, высокая) и выведите отформатированный ответ.
2. Реализуйте шаблон Проверка понимания (3-й шаблон).
3. Реализуйте 5-й шаблон Подумаем шаг за шагом, используя системный промпт.
4. Реализуйте 4-й шаблон Обратное взаимодействие, используя историю.

## Подключение

In [ ]:
from pathlib import Path
import sys

# При открытии из подпапки находим общий модуль конфигурации.
for candidate in [Path.cwd(), Path.cwd().parent]:
    if (candidate / "lab_runtime.py").exists():
        sys.path.insert(0, str(candidate))
        break

from lab_runtime import make_llm, json_result, ASSETS
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from IPython.display import display, Markdown, Image, Audio

llm = make_llm()
print("Модель:", llm.model_name)


## 1. Ответы при трёх температурах

In [ ]:
question = "Как применить искусственный интеллект для проверки музыкальных текстов? Предложи 3 подхода, до 100 слов."
answers = {}
for temperature in [0.1, 0.7, 1]:
    answer = make_llm(temperature=temperature).invoke(question).content
    answers[temperature] = answer
    display(Markdown(f"### Температура {temperature}\n\n{answer}"))


## 2. Проверка понимания

In [ ]:
for temperature, answer in answers.items():
    print(f"\nТемпература: {temperature}\n{answer}\n")

In [ ]:
understanding = ChatPromptTemplate.from_messages([
    ("system", "Ты преподаватель. Сначала дай определение, затем задай проверочный вопрос."),
    ("human", "Объясни {topic} в двух предложениях и задай один вопрос.")
]) | llm | StrOutputParser()
explanation = understanding.invoke({"topic": "RAG"})
print(explanation)
reply = "RAG ищет информацию в базе знаний и добавляет её в контекст запроса к модели."
feedback = llm.invoke(f"Объяснение и вопрос: {explanation}\nОтвет студента: {reply}\nОцени правильность кратко.").content
print("Ответ студента:", reply)
print("Проверка:", feedback)


## 3. Шаблон «Подумаем шаг за шагом»

In [ ]:
step_prompt = ChatPromptTemplate.from_messages([
    ("system", "Решай задачу последовательно. Покажи краткий план и проверяемые вычисления, затем итог."),
    ("human", "{task}")
]) | llm | StrOutputParser()
print(step_prompt.invoke({"task": "В тексте 1200 символов. Чанк — 400 символов, перекрытие — 100. Сколько чанков нужно? Покажи позиции."}))


## 4. Обратное взаимодействие с историей

In [ ]:
messages = [SystemMessage(content="Ты консультант. Перед рекомендацией уточни задачу пользователя, задавая по одному вопросу.")]
for user_text in [
    "Хочу создать музыкального помощника.",
    "Для независимых исполнителей, нужна помощь с текстами песен.",
    "Нужна проверка нежелательного содержания, теперь предложи решение в 3 пунктах."
]:
    messages.append(HumanMessage(content=user_text))
    response = llm.invoke(messages)
    messages.append(response)
    print("Пользователь:", user_text)
    print("Ассистент:", response.content, "\n")
assert len(messages) == 7


## Вывод

Сравнены три температуры. Реализованы проверка понимания, последовательное решение задачи и уточняющий диалог с сохранением истории.